In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════════════╗
# ║ lemmalog — does coreference need a second tier? (Colab, single cell)                      ║
# ║ Steps: deps → clone → Lean4 proofs → cargo tests → GLiNER ONNX → label your conversations  ║
# ║        → score gold: sieve abstention rate, per-tier accuracy, mention-pair MCC (sklearn)   ║
# ╚══════════════════════════════════════════════════════════════════════════════════════════╝
LEMMALOG_REPO   = "https://github.com/dhaneshkk/lemmalog"
LEMMALOG_BRANCH = "main"
GLINER_MODEL    = "onnx-community/gliner-multitask-large-v0.5"
# Your conversations, one JSON object per line: {"id": ..., "text": ...} or
# {"id": ..., "turns": [{"speaker": ..., "text": ...}]}. Upload it to /content.
CONVERSATIONS   = "/content/conversations.jsonl"
# The labelled file. Run once to get GOLD_TEMPLATE, fill in "clusters" for each line
# (see src/extract/coref_eval.rs for the format), save it as GOLD, run again.
GOLD            = "/content/coref_gold.jsonl"
SUGGEST         = False   # True adds the sieves' answers as "suggested_clusters" (anchors labellers: off by default)
WORK            = "/content/lemmalog_coref"

import os, sys, json, subprocess, glob
def sh(cmd, check=True, env=None, quiet=False):
    print(f"$ {cmd[:160]}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env={**os.environ, **(env or {})})
    out = (r.stdout + r.stderr).strip()
    if out and (not quiet or r.returncode): print(out[-6000:])
    if check and r.returncode: raise RuntimeError(f"failed: {cmd}")
    return r.stdout

# ── 0. dependencies ───────────────────────────────────────────────────────────────────────
sh(f"{sys.executable} -m pip install -q onnxruntime==1.20.1 huggingface_hub scikit-learn", quiet=True)
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    sh("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal", quiet=True)
if not os.path.exists(os.path.expanduser("~/.elan/bin/lean")):
    sh("curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:v4.15.0", quiet=True)
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]
import onnxruntime as _ort
ORT_DIR = os.path.join(os.path.dirname(_ort.__file__), "capi")
ORT_SO = sorted(glob.glob(os.path.join(ORT_DIR, "libonnxruntime.so*")))[-1]
ENV = {"ORT_LIB_LOCATION": ORT_DIR, "ORT_DYLIB_PATH": ORT_SO, "RUST_BACKTRACE": "0"}
os.environ.update(ENV)
FEATURES = "gliner-load-dynamic"

# ── 1. sources ────────────────────────────────────────────────────────────────────────────
os.makedirs(WORK, exist_ok=True)
LM = f"{WORK}/lemmalog"
if not os.path.exists(LM): sh(f"git clone -q --depth 1 -b {LEMMALOG_BRANCH} {LEMMALOG_REPO} {LM}")
sh(f"cd {LM} && git fetch -q --depth 1 origin {LEMMALOG_BRANCH} && git reset -q --hard FETCH_HEAD")
print("lemmalog at", sh(f"cd {LM} && git log -1 --format='%h %s'", quiet=True).strip())

# ── 2. Lean4: tier 2 touches only abstentions, the gate picks at most one, pairs partition ─
sh(f"cd {LM}/lean && lean Coref.lean")
print("Lean4: all theorems in lean/Coref.lean type-check ✔")

# ── 3. cargo tests: sieves (O7), link facts + retraction (O8), gated tier 2 (O9) ───────────
sh(f"cd {LM} && cargo test --release --features {FEATURES},mcp --test coref_test --test coref_memory_test 2>&1 | grep -E 'test result|FAILED|panicked'", env=ENV)
sh(f"cd {LM} && cargo test --release --features {FEATURES} --lib coref 2>&1 | grep -E 'test result|FAILED|panicked'", env=ENV)
sh(f"cd {LM} && cargo build -q --release --features {FEATURES} --example coref_eval", env=ENV)
BIN = f"{LM}/target/release/examples/coref_eval"

# ── 4. label: GLiNER finds the mentions in your conversations ──────────────────────────────
GOLD_TEMPLATE = GOLD.replace(".jsonl", ".template.jsonl")
if os.path.exists(CONVERSATIONS):
    from huggingface_hub import hf_hub_download
    d = os.path.expanduser("~/.lemmalog/models/gliner-multitask-large-v0.5")
    for f in ("tokenizer.json", "onnx/model.onnx"):
        hf_hub_download(GLINER_MODEL, f, local_dir=d)
    out = sh(f"{BIN} label {CONVERSATIONS}" + (" --suggest" if SUGGEST else ""), env={**ENV, "LEMMALOG_GLINER_DIR": d}, quiet=True)
    open(GOLD_TEMPLATE, "w").write(out)
    print(f"wrote {GOLD_TEMPLATE}: {len(out.splitlines())} conversation(s); fill in \"clusters\" and save as {GOLD}")
else:
    print(f"no {CONVERSATIONS}: upload your conversations to label them")

# ── 5. score ──────────────────────────────────────────────────────────────────────────────
target = GOLD if os.path.exists(GOLD) else f"{LM}/benchmarks/coref_gold.example.jsonl"
if target != GOLD:
    print(f"\n⚠ no {GOLD} yet: scoring the FORMAT EXAMPLE, which measures nothing about your data\n")
print(sh(f"{BIN} score {target}", env=ENV, quiet=True))
rep = json.loads(sh(f"{BIN} score {target} --json", env=ENV, quiet=True))

# MCC cross-check with scikit-learn on the same mention pairs
from sklearn.metrics import matthews_corrcoef
p = rep["pairs"]
y_true = [1] * p["tp"] + [0] * p["tn"] + [0] * p["fp"] + [1] * p["fn"]
y_pred = [1] * p["tp"] + [0] * p["tn"] + [1] * p["fp"] + [0] * p["fn"]
sk = matthews_corrcoef(y_true, y_pred) if y_true else 0.0
assert abs(sk - p["mcc"]) < 1e-9, (sk, p["mcc"])
print(f"mention-pair MCC {p['mcc']:.4f} (sklearn {sk:.4f}) ✔")

# ── 6. do you need tier 2? ────────────────────────────────────────────────────────────────
pron, ab, head = rep["pronouns"], rep["abstained"], rep["abstained_with_antecedent"]
for name, t in rep["tiers"].items():
    acc = t["correct"] / t["resolved"] if t["resolved"] else float("nan")
    print(f"  {name:<8} resolved {t['resolved']:>4}  accuracy {acc:.3f}")
print(f"sieves abstain on {ab}/{pron} pronouns ({100*ab/max(pron,1):.1f}%); "
      f"{head} of those have a gold antecedent, the most a second tier could add.")
if head == 0:
    print("→ tier 2 has nothing to win on these conversations: keep the sieves alone.")
else:
    print(f"→ tier 2 could lift pronoun recall by up to {100*head/max(pron,1):.1f} points; "
          "it is worth it only if its gated accuracy on those cases beats the cost of wrong links.")
